<a href="https://colab.research.google.com/github/YaminisriVellanki/24B11DS221_DriveReady_OPC_2026/blob/main/task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 4: Tools, Structured Output and Google Sheets

## Objective
Learn how AI agents use tools, extract structured data from natural-language
messages, integrate with Google Sheets, analyze tool execution traces,
and evaluate agent behavior using customer scenarios.

Business Case: Sweet Crumbs Home Bakery

In [2]:
!pip install -q langchain langchain-groq

In [1]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

Enter your Groq API Key: ··········


In [3]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [4]:
response = llm.invoke("Say Hello in one short sentence.")

print(response.content)

Hello!


In [40]:
from langchain.tools import tool

@tool
def get_bakery_price(item: str) -> str:
    """Find the price of a bakery item from the menu."""

    menu = {
        "plum cake": 500,
        "chocolate cake": 650,
        "cupcake": 80
    }

    item = item.lower().strip()

    if item in menu:
        return f"{item} costs Rs. {menu[item]}."

    return "The requested item is not available in the menu."


@tool
def check_order_requirement() -> str:
    """Check the advance notice requirement for bakery orders."""

    return "Orders require at least 24 hours advance notice."


tools = [get_bakery_price, check_order_requirement]

for t in tools:
    print("Tool Name:", t.name)
    print("Description:", t.description)
    print("-" * 50)

Tool Name: get_bakery_price
Description: Find the price of a bakery item from the menu.
--------------------------------------------------
Tool Name: check_order_requirement
Description: Check the advance notice requirement for bakery orders.
--------------------------------------------------


In [41]:
print(get_bakery_price.invoke({"item": "plum cake"}))

print(check_order_requirement.invoke({}))

plum cake costs Rs. 500.
Orders require at least 24 hours advance notice.


## Task 1: Creating Tools

In this task, I converted two business functions into LangChain tools using the `@tool` decorator.

I created:

- `get_bakery_price` to retrieve the price of bakery items.
- `check_order_requirement` to check the advance-order requirement.

Each tool contains a descriptive docstring, which explains its purpose to the AI agent. I tested the tools individually using `.invoke()`.

This demonstrates how existing Python functions can be converted into tools that an AI agent can call when required.

In [42]:
from pydantic import BaseModel, Field
from typing import Optional

class BakeryOrder(BaseModel):
    customer_name: Optional[str] = Field(
        default=None, description="Name of the customer"
    )
    item: Optional[str] = Field(
        default=None, description="Bakery item requested"
    )
    quantity: Optional[int] = Field(
        default=None, description="Quantity requested"
    )
    order_date: Optional[str] = Field(
        default=None, description="Requested date for the order"
    )
    eggless: Optional[bool] = Field(
        default=None, description="Whether an eggless item is requested"
    )

print("BakeryOrder schema created.")

BakeryOrder schema created.


In [43]:
structured_llm = llm.with_structured_output(BakeryOrder)

print("Structured output configured.")

Structured output configured.


In [44]:
customer_messages = [
    "Hey I'm Ravi, need 2 plum cakes on Dec 24, eggless pls.",
    "Anu here. Can I get three chocolate cakes tomorrow?",
    "Send 10 cupcakes for Rahul this Sunday.",
    "One eggless chocolate cake please, I'm Priya.",
    "Kiran wants 5 plum cakes on December 30."
]

for message in customer_messages:
    result = structured_llm.invoke(
        f"Extract bakery order information from this message: {message}"
    )

    print("Customer Message:", message)
    print(result.model_dump())
    print("=" * 60)

Customer Message: Hey I'm Ravi, need 2 plum cakes on Dec 24, eggless pls.
{'customer_name': 'Ravi', 'item': 'plum cake', 'quantity': 2, 'order_date': 'Dec 24', 'eggless': True}
Customer Message: Anu here. Can I get three chocolate cakes tomorrow?
{'customer_name': 'Anu', 'item': 'chocolate cake', 'quantity': 3, 'order_date': 'tomorrow', 'eggless': None}
Customer Message: Send 10 cupcakes for Rahul this Sunday.
{'customer_name': 'Rahul', 'item': 'cupcakes', 'quantity': 10, 'order_date': 'this Sunday', 'eggless': None}
Customer Message: One eggless chocolate cake please, I'm Priya.
{'customer_name': 'Priya', 'item': 'chocolate cake', 'quantity': 1, 'order_date': None, 'eggless': True}
Customer Message: Kiran wants 5 plum cakes on December 30.
{'customer_name': 'Kiran', 'item': 'plum cakes', 'quantity': 5, 'order_date': 'December 30', 'eggless': None}


## Task 2: Structured Output Using Pydantic

In this task, I used Pydantic to define a structured customer-order model.

The model contains customer name, item, quantity, order date, and eggless preference.

I used `with_structured_output()` to convert informal customer messages into structured information that follows the Pydantic schema.

I tested the implementation using five customer messages with different wording and formats.

This demonstrates how structured output makes customer information easier to validate, process, and store.

In [45]:
from google.colab import auth
import gspread
from google.auth import default

auth.authenticate_user()

credentials, _ = default()
gc = gspread.authorize(credentials)

SHEET_NAME = "Sweet Crumbs Bakery Orders"

try:
    spreadsheet = gc.open(SHEET_NAME)
    print("Spreadsheet already exists.")

except gspread.SpreadsheetNotFound:
    spreadsheet = gc.create(SHEET_NAME)
    print("New spreadsheet created.")

worksheet = spreadsheet.sheet1

if not worksheet.get_all_values():
    worksheet.append_row([
        "Customer Name",
        "Item",
        "Quantity",
        "Order Date",
        "Eggless"
    ])

print("Google Sheet connected successfully.")
print("Spreadsheet URL:", spreadsheet.url)

New spreadsheet created.
Google Sheet connected successfully.
Spreadsheet URL: https://docs.google.com/spreadsheets/d/1frptxvlWsUGpuC8Q-ZXpBqPoHK82WCPnjExb25G1plw


In [46]:
from langchain.tools import tool

@tool
def log_bakery_order(
    customer_name: str,
    item: str,
    quantity: int,
    order_date: str,
    eggless: bool
) -> str:
    """Save a customer bakery order to the Google Sheets order register."""

    worksheet.append_row([
        customer_name,
        item,
        quantity,
        order_date,
        eggless
    ])

    return "Order saved successfully."

In [47]:
result = log_bakery_order.invoke({
    "customer_name": "Ravi",
    "item": "Plum Cake",
    "quantity": 2,
    "order_date": "24 December",
    "eggless": True
})

print(result)

Order saved successfully.


In [48]:
records = worksheet.get_all_records()

for record in records:
    print(record)

## Task 3: Google Sheets Integration

In this task, I integrated Google Sheets with Python using `gspread`.

I authenticated my Google account through Google Colab and created or opened a spreadsheet for storing bakery orders.

I implemented the `log_bakery_order` tool to save customer details, item, quantity, order date, and eggless preference.

I tested the tool with a sample order and verified that the record was saved in the Google Sheet.

This demonstrates how tools can interact with external applications and store business information.

In [49]:
from langchain.agents import create_agent

good_agent = create_agent(
    model=llm,
    tools=tools
)

good_result = good_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the price of a chocolate cake?"
        }
    ]
})

for message in good_result["messages"]:
    if getattr(message, "tool_calls", None):
        print("Tool Calls:", message.tool_calls)

print("\nFinal Answer:")
print(good_result["messages"][-1].content)

Tool Calls: [{'name': 'get_bakery_price', 'args': {'item': 'chocolate cake'}, 'id': 'fc_ff983ac4-fdc4-44b6-b3a2-ae501d6f656a', 'type': 'tool_call'}]

Final Answer:
The chocolate cake is priced at **Rs. 650**.


In [50]:
from langchain.tools import tool

@tool("tool1")
def tool1(item: str) -> str:
    """does stuff"""
    return get_bakery_price.invoke({"item": item})


poorly_named_tools = [
    tool1,
    check_order_requirement
]

poor_agent = create_agent(
    model=llm,
    tools=poorly_named_tools
)

poor_result = poor_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the price of a chocolate cake?"
        }
    ]
})

for message in poor_result["messages"]:
    if getattr(message, "tool_calls", None):
        print("Tool Calls:", message.tool_calls)

print("\nFinal Answer:")
print(poor_result["messages"][-1].content)

Tool Calls: [{'name': 'tool1', 'args': {'item': 'chocolate cake'}, 'id': 'fc_067d47e7-2c9c-4a8d-8d37-7e10f805396b', 'type': 'tool_call'}]

Final Answer:
The chocolate cake is priced at **₹ 650**.


In [51]:
print("Original Tool")
print("Name:", get_bakery_price.name)
print("Description:", get_bakery_price.description)

print("\nRenamed Tool")
print("Name:", tool1.name)
print("Description:", tool1.description)

Original Tool
Name: get_bakery_price
Description: Find the price of a bakery item from the menu.

Renamed Tool
Name: tool1
Description: does stuff


## Task 4: Poor Tool Naming Experiment

In this task, I experimented with the effect of tool names and descriptions on AI agent tool selection.

First, I tested the agent using descriptive tool names and docstrings.

Then, I renamed one tool to `tool1` and changed its description to `does stuff`. I asked the same question again and inspected the tool calls.

I compared the agent's behavior between the two versions.

This experiment demonstrates why meaningful tool names and clear descriptions are important when building AI agents.

In [52]:
trace_agent = create_agent(
    model=llm,
    tools=tools
)

trace = trace_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the price of two chocolate cakes? Also tell me the order advance notice."
        }
    ]
})

print("Agent run completed.")

Agent run completed.


In [53]:
for message in trace["messages"]:
    message.pretty_print()

================================ Human Message =================================

What is the price of two chocolate cakes? Also tell me the order advance notice.
================================== Ai Message ==================================
Tool Calls:
  get_bakery_price (fc_ab4eedcc-4bb1-4f8e-901c-58add50ca0b0)
 Call ID: fc_ab4eedcc-4bb1-4f8e-901c-58add50ca0b0
  Args:
    item: chocolate cake
================================= Tool Message =================================
Name: get_bakery_price

chocolate cake costs Rs. 650.
================================== Ai Message ==================================
Tool Calls:
  check_order_requirement (fc_214949af-d3b2-47f4-a4ea-856bec234550)
 Call ID: fc_214949af-d3b2-47f4-a4ea-856bec234550
  Args:
================================= Tool Message =================================
Name: check_order_requirement

Orders require at least 24 hours advance notice.
================================== Ai Message ==================================

Two

In [54]:
from langchain_core.messages import AIMessage, ToolMessage

ai_message_count = 0
tool_call_count = 0
tool_response_count = 0

for message in trace["messages"]:

    if isinstance(message, AIMessage):
        ai_message_count += 1
        tool_call_count += len(message.tool_calls)

    elif isinstance(message, ToolMessage):
        tool_response_count += 1

print("AI Messages:", ai_message_count)
print("Tool Calls:", tool_call_count)
print("Tool Responses:", tool_response_count)

AI Messages: 3
Tool Calls: 2
Tool Responses: 2


## Task 5: Agent Trace Analysis

In this task, I executed an AI agent and inspected its execution trace.

I used `pretty_print()` to view the messages generated during execution, including AI messages, tool calls, and tool responses.

I counted the AI messages, tool calls, and tool responses in the trace.

This helped me understand the agent execution process, including how the agent decides to call tools, receives tool results, and generates its final answer.

The AI message count represents the messages present in the captured trace and is not necessarily the exact number of provider-level model calls.